# 01 — Pricing Snapshot (API-first)

This notebook explains:
- Why we use a pricing API as the *serving* source of truth
- How to build a local snapshot for fast runtime lookups
- What fields you should store (ungraded + PSA 7–10 if available)


## 1) Configure secrets
Create `config/secrets.yaml` from `config/secrets.example.yaml`.

Never commit real keys.

In [ ]:
import yaml, pandas as pd
from pathlib import Path

cfg = yaml.safe_load(open('config/config.yaml'))
print(cfg)

## 2) Build / update the snapshot
This runs the same code as the script, but lets you inspect outputs here.

In [ ]:
!python scripts/build_price_db_from_api.py

In [ ]:
import pandas as pd
import yaml
from pathlib import Path
cfg = yaml.safe_load(open('config/config.yaml'))
db = pd.read_csv(Path(cfg['processed_data'])/cfg['pricing']['api_price_db_filename'])
db.head()

## 3) Coverage sanity check
Commons usually have ungraded prices. PSA prices may be missing (sparse graded market).

In [ ]:
cols = ['market_price','psa_7_price','psa_8_price','psa_9_price','psa_10_price']
print(db[cols].notna().sum())

## Quick setup (optional)

If you haven't downloaded the pricing snapshot yet, you can use the Google Drive helper:

```bash
cp config/assets.yaml.example config/assets.yaml
# edit config/assets.yaml with your Drive link
python scripts/download_assets.py --asset price_db
```

This notebook will **gracefully skip** sections if the CSV isn't present.

In [ ]:
from pathlib import Path
import pandas as pd

price_path = Path('data/processed/pricing/complete_price_database.csv')
price_path.exists(), price_path

In [ ]:
import pandas as pd
from pathlib import Path

if not price_path.exists():
    raise FileNotFoundError(
        'Pricing snapshot not found. Run: python scripts/download_assets.py --asset price_db\n'
        'or build it with: python scripts/build_price_db_from_api.py'
    )

df = pd.read_csv(price_path)
df.head()

## Coverage checks

These checks answer:
- How many cards have ungraded market price?
- How many have PSA 7/8/9/10?
- How sparse is the table?

This is the kind of “data sanity” that makes a portfolio project credible.

In [ ]:
cols = ['market_price','psa_7_price','psa_8_price','psa_9_price','psa_10_price']
coverage = {c: df[c].notna().mean() for c in cols}
coverage

In [ ]:
import matplotlib.pyplot as plt

# Simple bar chart (matplotlib only)
plt.figure(figsize=(6,3))
plt.bar(list(coverage.keys()), list(coverage.values()))
plt.xticks(rotation=45, ha='right')
plt.ylabel('Fraction non-null')
plt.title('Price Field Coverage')
plt.tight_layout()
plt.show()

## Price distribution (ungraded)

We look at the ungraded market distribution to confirm that:
- commons cluster near $0-$1
- there is a long tail of expensive cards

You’ll use this later to build sensible UX (e.g., show cents for commons).

In [ ]:
import numpy as np

mp = df['market_price'].dropna()
mp.describe(percentiles=[.5,.9,.99])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,3))
plt.hist(mp.clip(upper=200), bins=50)
plt.xlabel('Market price (clipped at $200)')
plt.ylabel('Count')
plt.title('Ungraded Market Price Distribution')
plt.tight_layout()
plt.show()

## Spot-check: one card

Pick a card_id you care about (e.g., Base Set Charizard) and inspect all price fields.

In [ ]:
example_card_id = df['card_id'].dropna().iloc[0]
row = df[df['card_id'] == example_card_id].iloc[0]
row[['card_id','card_name','set_name','market_price','psa_7_price','psa_8_price','psa_9_price','psa_10_price']].to_dict()